# Colab: Fine-tune Gemma-4 E2B for ASL Top-50 q64 gloss classification

This notebook trains a **LoRA adapter** for `unsloth/gemma-4-E2B-it` on the existing ASL Top-50 q64 JSONL contract:

- `asl_unsloth_pose_train_q64_full_top50_train.jsonl`
- `asl_unsloth_pose_train_q64_full_top50_val.jsonl`
- `asl_unsloth_pose_train_q64_full_top50_test.jsonl`
- `asl_unsloth_pose_train_q64_full_top50_manifest.json`

Outputs:

1. Local LoRA adapter folder in Colab.
2. Hugging Face adapter repo under `AlexD281`.
3. Optional local validation metrics and predictions CSV.

Run on a Colab **L4/T4/A100 GPU**. Gemma-4 E2B LoRA should fit around 8-10GB VRAM per Unsloth docs.


## 0) Runtime checklist

1. Colab: `Runtime` → `Change runtime type` → GPU.
2. Hugging Face: accept gated model terms if required for Gemma-4.
3. Create a write-scoped HF token at <https://huggingface.co/settings/tokens>.
4. Upload the four Top-50 files from your local repo when prompted, or set `DATA_SOURCE = "hf"` if you have already uploaded them to a HF dataset repo.


In [ ]:
# Colab/Unsloth install. Restart runtime if Colab asks, then run from this cell again.
import os, sys
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    !pip install -U --no-cache-dir unsloth unsloth_zoo
    !pip install -U --no-cache-dir "transformers>=4.56.0" "trl>=0.21.0" "peft>=0.17.0" accelerate bitsandbytes datasets huggingface_hub pandas scikit-learn
else:
    print("Not running in Colab; skipping install cell.")


In [ ]:
import json
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi, login
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))


In [ ]:
# ---- Edit these if needed ----
HF_NAMESPACE = "AlexD281"
BASE_MODEL = "unsloth/gemma-4-E2B-it"
RUN_NAME = "asl-gemma4-e2b-q64-top50-lora"
ADAPTER_REPO_ID = f"{HF_NAMESPACE}/{RUN_NAME}"
DATASET_REPO_ID = f"{HF_NAMESPACE}/asl-top50-q64"

DATA_SOURCE = "upload"  # "upload" for Colab file picker, "hf" to load from DATASET_REPO_ID
WORK_DIR = Path("/content/asl_gemma4_e2b_top50")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "outputs" / RUN_NAME
PRED_DIR = WORK_DIR / "predictions"
for p in [DATA_DIR, OUTPUT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "asl_unsloth_pose_train_q64_full_top50_train.jsonl"
VAL_FILE = DATA_DIR / "asl_unsloth_pose_train_q64_full_top50_val.jsonl"
TEST_FILE = DATA_DIR / "asl_unsloth_pose_train_q64_full_top50_test.jsonl"
MANIFEST_FILE = DATA_DIR / "asl_unsloth_pose_train_q64_full_top50_manifest.json"

MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 8
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
LORA_RANK = 16
LORA_ALPHA = 16

print("Adapter repo:", ADAPTER_REPO_ID)
print("Dataset repo:", DATASET_REPO_ID)


In [ ]:
# Login to Hugging Face. Use a write-scoped token for push_to_hub.
from getpass import getpass

hf_token = os.environ.get("HF_TOKEN") or getpass("HF_TOKEN: ")
login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)
print(api.whoami()["name"])


In [ ]:
# Load the dataset files into Colab.
if DATA_SOURCE == "upload":
    from google.colab import files
    print("Upload train.jsonl, val.jsonl, test.jsonl, and manifest.json from data/processed/exports/.")
    uploaded = files.upload()
    for filename, payload in uploaded.items():
        target = DATA_DIR / filename
        target.write_bytes(payload)
        print("saved", target, target.stat().st_size)
elif DATA_SOURCE == "hf":
    from huggingface_hub import hf_hub_download
    for filename in [TRAIN_FILE.name, VAL_FILE.name, TEST_FILE.name, MANIFEST_FILE.name]:
        downloaded = hf_hub_download(repo_id=DATASET_REPO_ID, repo_type="dataset", filename=filename, token=hf_token)
        target = DATA_DIR / filename
        target.write_bytes(Path(downloaded).read_bytes())
        print("downloaded", target, target.stat().st_size)
else:
    raise ValueError(f"Unknown DATA_SOURCE={DATA_SOURCE!r}")

for path in [TRAIN_FILE, VAL_FILE, TEST_FILE, MANIFEST_FILE]:
    assert path.exists() and path.stat().st_size > 0, f"Missing or empty: {path}"

In [ ]:
# Optional: upload the dataset files to a private HF dataset repo for repeatable future Colab runs.
UPLOAD_DATASET_TO_HF = False
if UPLOAD_DATASET_TO_HF:
    api.create_repo(DATASET_REPO_ID, repo_type="dataset", private=True, exist_ok=True)
    for path in [TRAIN_FILE, VAL_FILE, TEST_FILE, MANIFEST_FILE]:
        api.upload_file(
            path_or_fileobj=str(path),
            path_in_repo=path.name,
            repo_id=DATASET_REPO_ID,
            repo_type="dataset",
        )
        print("uploaded", path.name, "->", DATASET_REPO_ID)


In [ ]:
# Inspect the data contract.
manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
labels = manifest.get("labels") or manifest.get("glosses") or []
labels = [str(x).strip().lower() for x in labels]
print("labels:", len(labels), labels[:10])

with TRAIN_FILE.open("r", encoding="utf-8") as f:
    sample = json.loads(next(f))
print(sample.keys())
print("instruction:", sample.get("instruction", "")[:160])
print("input chars:", len(sample.get("input", "")))
print("output:", sample.get("output"))


In [ ]:
# Load JSONL splits with datasets.
data_files = {"train": str(TRAIN_FILE), "validation": str(VAL_FILE), "test": str(TEST_FILE)}
raw = load_dataset("json", data_files=data_files)
raw


In [ ]:
# Convert instruction/input/output rows into Gemma chat text.
# We train on assistant responses only below, so the q64 pose prompt is context and the gloss is the supervised target.
def row_to_messages(row):
    instruction = str(row.get("instruction", "Classify this compact ASL pose encoding into its WLASL gloss. Reply with only the gloss word."))
    pose_input = str(row.get("input", ""))
    output = str(row.get("output", "")).strip().lower()
    user_text = f"{instruction}\n\n{pose_input}"
    return [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": output},
    ]

# Keep a separate prompt-only formatter for evaluation generation.
def row_to_prompt_messages(row):
    instruction = str(row.get("instruction", "Classify this compact ASL pose encoding into its WLASL gloss. Reply with only the gloss word."))
    pose_input = str(row.get("input", ""))
    return [{"role": "user", "content": f"{instruction}\n\n{pose_input}"}]


In [ ]:
# HF / Unsloth download stability preflight.
# Run this BEFORE importing `unsloth`.
# The 120s error you saw is Unsloth's optional environment/statistics ping,
# not the Gemma-4 model download. Disable that ping but keep Hugging Face
# enabled for the actual Gemma-4 model files.
import os

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ.pop("UNSLOTH_USE_MODELSCOPE", None)  # keep using Hugging Face, not ModelScope

# Longer HF Hub timeouts help Colab when the real model download is slow.
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "600")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

print("Disabled Unsloth statistics check; Gemma-4 will still download from Hugging Face.")



In [ ]:
from unsloth import FastModel

try:
    model, tokenizer = FastModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        full_finetuning=False,
        token=hf_token,
    )
except TimeoutError as exc:
    raise RuntimeError(
        "Unsloth timed out before model loading. Restart the runtime, run the "
        "'HF / Unsloth download stability preflight' cell, confirm it prints "
        "that UNSLOTH_DISABLE_STATISTICS is enabled, then re-run this cell. "
        "This keeps the actual Gemma-4 download on Hugging Face."
    ) from exc

model = FastModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    target_modules="all-linear",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print("Loaded base model and tokenizer:", BASE_MODEL)
print("Tokenizer class:", type(tokenizer).__name__)



In [ ]:
if "tokenizer" not in globals():
    raise RuntimeError("tokenizer is not defined. Run the previous 'FastModel.from_pretrained' cell first; it creates model and tokenizer.")

def formatting_prompts_func(examples):
    texts = []
    for instruction, pose_input, output in zip(examples["instruction"], examples["input"], examples["output"]):
        messages = [
            {"role": "user", "content": f"{instruction}\n\n{pose_input}"},
            {"role": "assistant", "content": str(output).strip().lower()},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        texts.append(text)
    return {"text": texts}

train_dataset = raw["train"].map(formatting_prompts_func, batched=True, remove_columns=raw["train"].column_names)
val_dataset = raw["validation"].map(formatting_prompts_func, batched=True, remove_columns=raw["validation"].column_names)
print(train_dataset[0]["text"][:1000])


In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir=str(OUTPUT_DIR),
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.03,
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=SEED,
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

# Mask user prompt tokens so loss is applied only to assistant gloss responses.
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)


In [ ]:
# Start/resume training.
trainer_stats = trainer.train(resume_from_checkpoint=False)
trainer_stats


In [ ]:
# Save local LoRA adapter and tokenizer.
ADAPTER_DIR = OUTPUT_DIR / "adapter"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("saved adapter:", ADAPTER_DIR)
print(sorted(p.name for p in ADAPTER_DIR.iterdir()))


In [ ]:
# Push LoRA adapter to Hugging Face.
api.create_repo(ADAPTER_REPO_ID, repo_type="model", private=True, exist_ok=True)
model.push_to_hub(ADAPTER_REPO_ID, token=hf_token)
tokenizer.push_to_hub(ADAPTER_REPO_ID, token=hf_token)
print("pushed adapter to", ADAPTER_REPO_ID)


In [ ]:
# Quick validation/test generation loop.
FastModel.for_inference(model)

label_set = {str(x).strip().lower() for x in labels}

def normalize_gloss(text):
    text = str(text).strip().lower()
    text = re.sub(r"^`+|`+$", "", text)
    text = re.sub(r"[^a-z0-9_ -]+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    # Use first line / delimiter-separated candidate, matching project evaluator behavior.
    pieces = [p.strip() for p in re.split(r"[\n,;:]+", text) if p.strip()]
    return pieces[0] if pieces else ""

@torch.inference_mode()
def predict_record(row):
    messages = row_to_prompt_messages(row)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
        use_cache=True,
    )
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    raw_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return raw_text, normalize_gloss(raw_text)

def evaluate_split(split_name="validation", max_samples=None):
    rows = list(raw[split_name])
    if max_samples:
        rows = rows[:max_samples]
    y_true, y_pred, raw_outputs = [], [], []
    for i, row in enumerate(rows, start=1):
        raw_text, pred = predict_record(row)
        expected = normalize_gloss(row["output"])
        y_true.append(expected)
        y_pred.append(pred if pred in label_set else "__invalid__")
        raw_outputs.append(raw_text)
        if i % 25 == 0:
            print(f"{split_name}: {i}/{len(rows)}")
    acc = accuracy_score(y_true, y_pred)
    df = pd.DataFrame({"expected": y_true, "predicted": y_pred, "raw_model_output": raw_outputs})
    out_csv = PRED_DIR / f"{split_name}_predictions.csv"
    df.to_csv(out_csv, index=False)
    print(split_name, "accuracy", acc, "predictions", out_csv)
    print(classification_report(y_true, y_pred, zero_division=0))
    return df, acc

val_predictions, val_accuracy = evaluate_split("validation")


In [ ]:
# Optional held-out test evaluation. Keep disabled until you are ready to report metrics.
RUN_TEST_EVAL = False
if RUN_TEST_EVAL:
    test_predictions, test_accuracy = evaluate_split("test")


In [ ]:
# Optional: save a merged 16-bit model locally in Colab. This can be slow/heavy; the export notebook also supports this.
SAVE_MERGED_LOCAL = False
MERGED_DIR = WORK_DIR / "merged_16bit" / RUN_NAME
if SAVE_MERGED_LOCAL:
    model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
    print("merged model saved:", MERGED_DIR)
